# Datenbasis für die Kfz-Haftpflichtanalyse

Die Frequenz- und Betragstabellen von freMTPL2 erfassen unterschiedliche Einheiten: eine Zeile je Vertrag beziehungsweise je erfasstem Schadenbetrag. Für die Analyse werden positive Betragszeilen je Vertrag gezählt und summiert. Der Abgleich prüft, ob die mitgelieferte Schadenanzahl von dieser Zielgröße abweicht und wie sich hypothetische Kappungen auf den Bestand auswirken. [Validierungsregeln](../docs/validierung.md) · [Reproduktion der Datenbasis](../docs/reproduktion.md)

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/data_contract.json").is_file())

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

report = read_json(ROOT / "reports/s1/preparation_report.json")

## Zielgrößen je Vertrag

Die Exponierung misst die beobachtete Vertragsdauer in Jahren. Verträge ohne positive Betragszeile gehen mit Anzahl und Betrag null in die Analyse ein. Die Frequenz bezieht sich auf erfasste positive Schadenzeilen. Gemeldete Ansprüche ohne erfassten positiven Betrag sind darin nicht abgebildet.

In [2]:
display(pd.DataFrame(report["contract"]["units"].items(),
                     columns=["Spalte", "Bedeutung / Einheit"]))
print("Quelle:", report["source"]["version"], report["source"]["commit"])

,Spalte,Bedeutung / Einheit
0,ClaimAmount,Erfasste historische EUR je positiver Schadenz...
1,ClaimNb,Mitgelieferte Anzahl je Vertragsperiode
2,Exposure,Vertragsjahre
3,IDpol,Quellschlüssel des Vertrags; kein Haushalts- o...
4,claim_row_id,Technische Zeilenherkunft: SHA-256 der Rohdate...
5,recorded_loss_eur,"Summe erfasster EUR je Vertragsperiode, einsch..."
6,recorded_positive_count,Anzahl verknüpfter positiver Schadenzeilen je ...


Quelle: 1.2-1 227fb56b8734bdb7c0327a41180e01d2ddaeaf26


## Vertragsjahre, Anzahlen und erfasste EUR im Gesamtbestand

Der Abgleich verwendet den Vertragsschlüssel `IDpol`. Die Kreuztabelle zeigt, wie viele Verträge auf jede Kombination aus mitgelieferter Anzahl und Anzahl positiver Betragszeilen entfallen.

In [3]:
labels = {"policies": "Vertragszeilen", "exposure_years": "Exponierung [Vertragsjahre]",
          "supplied_claims": "Mitgelieferte Anzahl",
          "recorded_positive_claims": "Rekonstruierte positive Schadenzeilen",
          "recorded_loss_eur": "Erfasster Schadenbetrag [EUR]"}
reconciliation = report["reconciliation"]
display(pd.DataFrame({"Größe": [labels[k] for k in labels],
                      "Erhalten": [reconciliation["retained"][k] for k in labels]}))
display(pd.DataFrame([
    {"Prüfung": "Verträge ohne Schadenzeilen", **reconciliation["anti_joins"]["policy_without_claim_rows"]},
    {"Prüfung": "Verträge mit unterschiedlicher Anzahl", **reconciliation["supplied_vs_derived_mismatch"]}
]).rename(columns=labels))
display(pd.DataFrame([reconciliation["anti_joins"]["claim_rows_without_policy"]])
        .rename(columns={"claim_rows": "Schadenzeilen ohne Vertrag", "policies": "Quellschlüssel",
                         "recorded_loss_eur": "Nicht verknüpfter Betrag [EUR]"}))
print("Wiederholte Zeilen mit gleichem Vertrag und Betrag:",
      reconciliation["exact_repeat_claim_rows_beyond_first_retained"])
display(pd.DataFrame(reconciliation["count_cross_table"])
        .rename(columns={"supplied": "Mitgelieferte Anzahl", "derived": "Rekonstruierte Anzahl",
                         "policies": "Vertragszeilen"}))

,Größe,Erhalten
0,Vertragszeilen,6.779910e+05
1,Exponierung [Vertragsjahre],3.584828e+05
2,Mitgelieferte Anzahl,2.644400e+04
3,Rekonstruierte positive Schadenzeilen,2.644400e+04
4,Erfasster Schadenbetrag [EUR],5.990922e+07


,Prüfung,Exponierung [Vertragsjahre],Vertragszeilen,Erfasster Schadenbetrag [EUR],Rekonstruierte positive Schadenzeilen,Mitgelieferte Anzahl
0,Verträge ohne Schadenzeilen,341204.907202,653047,0.0,0,0
1,Verträge mit unterschiedlicher Anzahl,0.000000,0,0.0,0,0


,Schadenzeilen ohne Vertrag,Quellschlüssel,Nicht verknüpfter Betrag [EUR]
0,0,0,0.0


Wiederholte Zeilen mit gleichem Vertrag und Betrag: 235


,Rekonstruierte Anzahl,Vertragszeilen,Mitgelieferte Anzahl
0,0,653047,0
1,1,23571,1
2,2,1298,2
3,3,62,3
4,4,5,4
5,5,2,5
6,6,1,6
7,8,1,8
8,9,1,9
9,11,2,11


## Seltene Merkmalswerte und große Beträge im Gesamtbestand

Kurze Exponierungen, ungewöhnliche Alterswerte und B12 beschreiben Teilbestände. Die Tabellen zeigen deren Umfang und Schadenvolumen. Gleiche Beträge zum selben Vertrag bleiben erhalten, weil eine eindeutige Schaden-ID fehlt.

In [4]:
audit = report["impact_audit"]
flag_labels = {"exposure_above_one": "Exponierung > 1 Jahr",
               "exposure_below_one_month": "Exponierung < 1/12 Jahr",
               "driver_age_above_90": "Fahreralter > 90",
               "vehicle_age_above_40": "Fahrzeugalter > 40",
               "driver_age_100": "Fahreralter = 100",
               "vehicle_age_100": "Fahrzeugalter = 100",
               "count_above_four": "Schadenanzahl > 4",
               "bonus_malus_above_150": "Bonus-Malus > 150",
               "brand_B12": "Markenklasse B12",
               "brand_B12_new_vehicle": "B12 und Fahrzeugalter = 0"}
display(pd.DataFrame([{ "Prüfgruppe": flag_labels[name], **values}
                      for name, values in audit["policy_flags"].items()]).rename(columns=labels))
display(pd.DataFrame(audit["ranges"]).T.rename(columns={"min": "Minimum", "max": "Maximum"}))
display(pd.DataFrame(audit["claim_amount_quantiles_eur"].items(),
                     columns=["Quantil", "Schadenbetrag [EUR]"]))

,Prüfgruppe,Exponierung [Vertragsjahre],Vertragszeilen,Erfasster Schadenbetrag [EUR],Rekonstruierte positive Schadenzeilen,Mitgelieferte Anzahl
0,Bonus-Malus > 150,95.128197,209,132676.41,54,54
1,Markenklasse B12,64807.398718,166022,8747094.58,4200,4200
2,B12 und Fahrzeugalter = 0,9560.637042,37069,1139446.39,578,578
3,Schadenanzahl > 4,2.380000,8,167225.70,71,71
4,Fahreralter = 100,1.730000,3,0.00,0,0
5,Fahreralter > 90,315.938204,401,11721.74,16,16
6,Exponierung > 1 Jahr,1363.340000,1224,87927.38,54,54
7,Exponierung < 1/12 Jahr,6317.415463,114792,3464224.14,1043,1043
8,Fahrzeugalter = 100,15.150000,25,0.00,0,0
9,Fahrzeugalter > 40,154.518197,230,4426.57,3,3


,Maximum,Minimum
BonusMalus,230.00,50.000000
ClaimNb,16.00,0.000000
Density,27000.00,1.000000
DrivAge,100.00,18.000000
Exposure,2.01,0.002732
VehAge,100.00,0.000000


,Quantil,Schadenbetrag [EUR]
0,0,1.000000e+00
1,0.5,1.172000e+03
2,0.9,2.767604e+03
3,0.95,4.765093e+03
4,0.99,1.645122e+04
5,0.999,1.522232e+05
6,1,4.075401e+06


## Wie Kappungen Exponierung, Anzahl und erfasste EUR verändern

Eine Obergrenze für die Dauer ändert den Nenner der Jahresrate. Eine Kappung von Anzahlen oder Beträgen ändert dagegen die Zielgröße. Die Szenarien zeigen die Auswirkungen getrennt.

In [5]:
display(pd.DataFrame([audit["hypothetical_exposure_cap_one"]]).rename(columns={
    "removed_policy_years": "Bei Kappung auf 1 entfernte Vertragsjahre",
    "remaining_policy_years": "Verbleibende Vertragsjahre",
    "unchanged_claims": "Unveränderte Schadenzeilen", "unchanged_loss_eur": "Unveränderter Betrag [EUR]"}))
display(pd.DataFrame([audit["hypothetical_count_cap_four"]]).rename(columns={
    "removed_counts": "Bei Kappung auf 4 entfernte Anzahlen",
    "remaining_counts": "Verbleibende Anzahlen", "loss_allocation": "Eurozuordnung"}))
display(pd.DataFrame([{ "Kappung je Schaden [EUR]": cap, **values}
                      for cap, values in audit["hypothetical_claim_amount_caps_eur"].items()])
        .rename(columns={"claim_rows": "Betroffene Schadenzeilen", "policies": "Betroffene Verträge",
                         "recorded_loss_eur": "Betrag der betroffenen Schäden [EUR]",
                         "removed_loss_eur": "Entfernter Betrag [EUR]",
                         "remaining_loss_eur": "Verbleibender Gesamtbetrag [EUR]"}))

,Verbleibende Vertragsjahre,Bei Kappung auf 1 entfernte Vertragsjahre,Unveränderte Schadenzeilen,Unveränderter Betrag [EUR]
0,358343.495463,139.34,26444,59909216.5


,Eurozuordnung,Verbleibende Anzahlen,Bei Kappung auf 4 entfernte Anzahlen
0,Nicht definiert: keine willkürliche Auswahl zu...,26405,39


,Kappung je Schaden [EUR],Betroffene Schadenzeilen,Betroffene Verträge,Betrag der betroffenen Schäden [EUR],Verbleibender Gesamtbetrag [EUR],Entfernter Betrag [EUR]
0,100000,41,41,14707020.45,49302196.05,10607020.45
1,1000000,3,3,6779630.56,56129585.94,3779630.56
2,25000,177,177,21013178.15,43321038.35,16588178.15


## Datenbasis für Frequenz und Schadenhöhe

Die Primäranalyse verwendet die rekonstruierte Anzahl und die ungekappte Summe der positiven Beträge. Die Exponierung bleibt in Vertragsjahren erhalten. Die [feste Vertragsaufteilung](02_validation_contracts.ipynb) bildet die Grundlage für die Modellvergleiche.